In [1]:
import pandas as pd
import numpy as np

In [5]:
# =============================================================
# STAGE 3 — RISK SCORING FRAMEWORK
# =============================================================
# NOTE: order_customer_id is used as vendor proxy — dataset has no dedicated supplier identifier.

df = pd.read_csv(r'C:\Users\Ashok Thakur\Desktop\supply_chain_project\data\processed\supply_chain_clean.csv')

VENDOR_COL   = 'order_customer_id'
CATEGORY_COL = 'category_name'
SPEND_COL    = 'spend_per_order'

In [6]:
# ------------------------------------------------------------------
# Task 6: Herfindahl-Hirschman Index per commodity category
# HHI = sum of squared market share of each vendor within a category
# HHI → 1.0 = monopoly (critical concentration risk)
# ------------------------------------------------------------------

category_spend = (df.groupby([CATEGORY_COL, VENDOR_COL])[SPEND_COL]
                  .sum()
                  .reset_index())
category_spend.columns = ['category', 'vendor', 'vendor_spend']

category_total = (category_spend.groupby('category')['vendor_spend']
                  .sum()
                  .reset_index()
                  .rename(columns={'vendor_spend': 'category_total'}))

category_spend = category_spend.merge(category_total, on='category')
category_spend['market_share']  = (category_spend['vendor_spend'] /
                                    category_spend['category_total'])
category_spend['share_squared'] = category_spend['market_share'] ** 2

hhi = (category_spend.groupby('category')['share_squared']
       .sum()
       .reset_index()
       .rename(columns={'share_squared': 'hhi_score'}))

print("Top 10 categories by HHI concentration:")
print(hhi.sort_values('hhi_score', ascending=False).head(10))

Top 10 categories by HHI concentration:
               category  hhi_score
1       As Seen on  TV!   0.024409
41    Strength Training   0.022212
4            Basketball   0.022010
39               Soccer   0.016629
23    Golf Bags & Carts   0.016493
35     Men's Golf Clubs   0.013371
49   Women's Golf Clubs   0.011640
32             Lacrosse   0.007502
31     Kids' Golf Clubs   0.006689
19  Fitness Accessories   0.005542


In [7]:
# ------------------------------------------------------------------
# Task 7: Lead time variance score per vendor
# Std deviation of lead_time_days, min-max normalised to [0,1]
# High variance = unpredictable supplier = higher risk
# ------------------------------------------------------------------

lead_stats = (df.groupby(VENDOR_COL)['lead_time_days']
              .agg(['mean', 'std'])
              .reset_index())
lead_stats.columns = ['vendor', 'avg_lead_time', 'lead_time_std']
lead_stats['lead_time_std'] = lead_stats['lead_time_std'].fillna(0)

lt_min = lead_stats['lead_time_std'].min()
lt_max = lead_stats['lead_time_std'].max()
lead_stats['lead_time_variance_score'] = ((lead_stats['lead_time_std'] - lt_min) /
                                           (lt_max - lt_min + 1e-9))

In [8]:
# ------------------------------------------------------------------
# Task 8: Late delivery rate per vendor
# Proportion of late orders per vendor — proxy for supplier health
# ------------------------------------------------------------------

late_rate = (df.groupby(VENDOR_COL)['late_delivery_flag']
             .mean()
             .reset_index()
             .rename(columns={VENDOR_COL: 'vendor',
                               'late_delivery_flag': 'late_delivery_rate'}))

In [9]:
# ------------------------------------------------------------------
# Task 9: Composite risk score
# Score = 0.4 × lead time variance + 0.6 × late delivery rate
# Normalised to [0,1]; higher = riskier vendor
# ------------------------------------------------------------------

vendor_risk = lead_stats.merge(late_rate, on='vendor')

vendor_risk['composite_risk_score'] = (
    0.4 * vendor_risk['lead_time_variance_score'] +
    0.6 * vendor_risk['late_delivery_rate']
)

vmin = vendor_risk['composite_risk_score'].min()
vmax = vendor_risk['composite_risk_score'].max()
vendor_risk['composite_risk_score'] = ((vendor_risk['composite_risk_score'] - vmin) /
                                        (vmax - vmin + 1e-9))

In [10]:
# ------------------------------------------------------------------
# Task 10: Risk band classification
# 4 tiers: Low / Medium / High / Critical — Power BI slicer dimension
# ------------------------------------------------------------------

def classify_risk(score):
    if score < 0.25:   return 'Low'
    elif score < 0.50: return 'Medium'
    elif score < 0.75: return 'High'
    else:              return 'Critical'

vendor_risk['risk_band'] = vendor_risk['composite_risk_score'].apply(classify_risk)

print("\nRisk band distribution:")
print(vendor_risk['risk_band'].value_counts())
print("\nSample output:")
print(vendor_risk.head())


Risk band distribution:
risk_band
High        9990
Low         5334
Medium      4547
Critical     773
Name: count, dtype: int64

Sample output:
   vendor  avg_lead_time  lead_time_std  lead_time_variance_score  \
0       1       6.000000       0.000000                  0.000000   
1       2       4.500000       1.080123                  0.311805   
2       3       2.055556       1.924218                  0.555474   
3       4       2.285714       1.857565                  0.536233   
4       5       3.142857       0.377964                  0.109109   

   late_delivery_rate  composite_risk_score risk_band  
0            1.000000              0.600000      High  
1            0.300000              0.304722    Medium  
2            0.000000              0.222190       Low  
3            0.785714              0.685922      High  
4            0.714286              0.472215    Medium  


In [13]:
# ------------------------------------------------------------------
# Task 11: Export risk scores CSV
# Feeds into MySQL as vendor_risk_scores table
# Also feeds vw_vendor_risk_master in 04_reporting_views.sql
# ------------------------------------------------------------------

vendor_risk.to_csv(r'C:\Users\Ashok Thakur\Desktop\supply_chain_project\data\processed\vendor_risk_scores.csv', index=False)
print("\nExported: data/processed/vendor_risk_scores.csv")
print("Shape:", vendor_risk.shape)


Exported: data/processed/vendor_risk_scores.csv
Shape: (20644, 7)
